# EnergyLens-AI - Notebook 2: Data Cleaning

In the EDA notebook I found a few issues I need to deal with before modeling:
1. Right-skewed distribution with some extreme outliers
2. Missing values in daily stats (like `energy_std` when `energy_count` is 1)
3. A few missing values in weather columns (e.g. `cloudCover`)
4. Large memory usage because of default dtypes

In this notebook I clean and merge the three raw datasets into one master dataset, and also downcast the dtypes so it doesn't eat up all my RAM.

I didn't just drop missing values or clip outliers without thinking about it - I checked why each value was missing first (for example, setting standard deviation to 0 when there's only 1 reading makes sense mathematically), and downcasting types cut memory usage a lot, which matters since I'm working with 3.5+ million rows on my laptop.


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 6)

# Check if running in Colab or locally
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✅ Environment checked. Running in Colab: {IN_COLAB}")


✅ Environment checked. Running in Colab: True


## 2. Loading the Data

I load either from local file paths or via upload in Colab, depending on where this is running.


In [2]:
if IN_COLAB:
    from google.colab import files
    print("📤 Please upload informations_households.csv")
    uploaded_hh = files.upload()
    hh_path = list(uploaded_hh.keys())[0]

    print("\n📤 Please upload weather_daily_darksky.csv")
    uploaded_weather = files.upload()
    weather_path = list(uploaded_weather.keys())[0]

    print("\n📤 Please upload daily_dataset.csv")
    uploaded_daily = files.upload()
    daily_path = list(uploaded_daily.keys())[0]
else:
    hh_path = '../data/raw/informations_households.csv'
    weather_path = '../data/raw/weather_daily_darksky.csv'
    daily_path = '../data/raw/daily_dataset.csv'

    # Confirm local files exist
    for path in [hh_path, weather_path, daily_path]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing raw data file: {path}. Please place it in data/raw/")

print("✅ File paths mapped successfully.")


📤 Please upload informations_households.csv


Saving informations_households.csv to informations_households.csv

📤 Please upload weather_daily_darksky.csv


Saving weather_daily_darksky.csv to weather_daily_darksky.csv

📤 Please upload daily_dataset.csv


Saving daily_dataset.csv to daily_dataset.csv
✅ File paths mapped successfully.


In [3]:
# Load the datasets
print("Loading households...")
households = pd.read_csv(hh_path)
print("Loading weather...")
weather = pd.read_csv(weather_path)
print("Loading daily energy consumption (this may take a few seconds)...")
daily = pd.read_csv(daily_path)

print(f"\n📊 Initial shapes:")
print(f"  Households: {households.shape}")
print(f"  Weather: {weather.shape}")
print(f"  Daily Consumption: {daily.shape}")


Loading households...
Loading weather...
Loading daily energy consumption (this may take a few seconds)...

📊 Initial shapes:
  Households: (5566, 5)
  Weather: (882, 32)
  Daily Consumption: (3510433, 9)


## 3. Missing Values

Checking missing values column by column and treating each one based on what actually makes sense for that column, instead of just dropping everything.


In [4]:
# Household metadata - drop rows with missing tariff/ACORN
print("🔍 Checking households missing values:")
print(households.isnull().sum())

# Drop households missing tariff or ACORN group since we can't use them for segment-level analysis
initial_hh_count = len(households)
households = households.dropna(subset=['stdorToU', 'Acorn'])
print(f"Dropped {initial_hh_count - len(households)} households with missing metadata.")


🔍 Checking households missing values:
LCLid            0
stdorToU         0
Acorn            0
Acorn_grouped    0
file             0
dtype: int64
Dropped 0 households with missing metadata.


In [5]:
# Weather data - fill missing values
print("🔍 Checking weather missing values (top 5):")
print(weather.isnull().sum().sort_values(ascending=False).head(5))

# Weather is a continuous time series, so interpolation works well and avoids sudden jumps
weather_cols_to_fill = ['cloudCover', 'uvIndex', 'uvIndexTime']
for col in weather_cols_to_fill:
    if col in weather.columns:
        null_count = weather[col].isnull().sum()
        if null_count > 0:
            weather[col] = weather[col].interpolate(method='linear').ffill().bfill()
            print(f"  Imputed {null_count} missing values in '{col}' using linear interpolation.")

# Verify no nulls remain in weather
assert weather.isnull().sum().sum() == 0, "Weather data still contains missing values!"
print("✅ Weather missing values resolved.")


🔍 Checking weather missing values (top 5):
cloudCover        1
uvIndex           1
uvIndexTime       1
temperatureMax    0
icon              0
dtype: int64
  Imputed 1 missing values in 'cloudCover' using linear interpolation.
  Imputed 1 missing values in 'uvIndex' using linear interpolation.
  Imputed 1 missing values in 'uvIndexTime' using linear interpolation.
✅ Weather missing values resolved.


In [6]:
# Daily consumption - check missing values
print("🔍 Checking daily energy consumption missing values:")
print(daily.isnull().sum())

# Check rows where energy_mean is missing
all_null_energy = daily[daily['energy_mean'].isnull()]
print(f"\nRows with missing energy statistics: {len(all_null_energy)}")

# Drop these rows - no actual consumption reading for that day
daily = daily.dropna(subset=['energy_mean'])
print(f"Dropped {len(all_null_energy)} rows with fully missing energy statistics.")

# Now look at remaining missing values (usually in energy_std)
print(f"Remaining nulls after dropping empty rows:")
print(daily.isnull().sum())


🔍 Checking daily energy consumption missing values:
LCLid                0
day                  0
energy_median       30
energy_mean         30
energy_max          30
energy_count         0
energy_std       11331
energy_sum          30
energy_min          30
dtype: int64

Rows with missing energy statistics: 30
Dropped 30 rows with fully missing energy statistics.
Remaining nulls after dropping empty rows:
LCLid                0
day                  0
energy_median        0
energy_mean          0
energy_max           0
energy_count         0
energy_std       11301
energy_sum           0
energy_min           0
dtype: int64


Why is `energy_std` missing? Standard deviation needs at least 2 readings to make sense. If a household has only 1 reading in a day (`energy_count == 1`), std is undefined and shows up as `NaN`.

I filled these with `0.0` since there's no variance when there's only one value.


In [7]:
# Treat energy_std missing values
std_null_count = daily['energy_std'].isnull().sum()
if std_null_count > 0:
    # Check if these rows have energy_count == 1
    sample_null_std = daily[daily['energy_std'].isnull()]['energy_count'].value_counts()
    print("Value counts of energy_count when energy_std is null:")
    print(sample_null_std)

    daily['energy_std'] = daily['energy_std'].fillna(0.0)
    print(f"✅ Filled {std_null_count} missing values in 'energy_std' with 0.0")

# Confirm no null values remain in daily
assert daily.isnull().sum().sum() == 0, "Daily dataset still contains missing values!"
print("✅ Daily consumption missing values resolved.")


Value counts of energy_count when energy_std is null:
energy_count
1    11301
Name: count, dtype: int64
✅ Filled 11301 missing values in 'energy_std' with 0.0
✅ Daily consumption missing values resolved.


## 4. Outliers

Outliers in energy consumption can come from two places:
1. Sensor/measurement errors (e.g. negative energy, which isn't physically possible)
2. Real but extreme usage (e.g. a very cold day causing high heating use)

Instead of dropping outlier rows (which would create gaps in the time series and hurt any time-based model), I capped them at the 99.5th percentile.

I compared IQR and Z-score first to decide on a threshold.


In [8]:
# Check energy_mean outliers
energy_data = daily['energy_mean']

# Method A: IQR
Q1 = energy_data.quantile(0.25)
Q3 = energy_data.quantile(0.75)
IQR = Q3 - Q1
lower_bound_iqr = Q1 - 1.5 * IQR
upper_bound_iqr = Q3 + 1.5 * IQR

# Method B: Z-score (3 standard deviations)
mean_val = energy_data.mean()
std_val = energy_data.std()
upper_bound_z = mean_val + 3 * std_val
lower_bound_z = mean_val - 3 * std_val

print(f"📊 Outlier Detection Thresholds (energy_mean):")
print(f"  IQR bounds: {lower_bound_iqr:.4f} to {upper_bound_iqr:.4f}")
print(f"  Z-score bounds (3 std): {lower_bound_z:.4f} to {upper_bound_z:.4f}")

# Count outliers
outliers_iqr = (energy_data < lower_bound_iqr) | (energy_data > upper_bound_iqr)
outliers_z = (energy_data < lower_bound_z) | (energy_data > upper_bound_z)

print(f"\n  IQR Outliers: {outliers_iqr.sum()} ({outliers_iqr.mean()*100:.2f}%)")
print(f"  Z-score Outliers: {outliers_z.sum()} ({outliers_z.mean()*100:.2f}%)")


📊 Outlier Detection Thresholds (energy_mean):
  IQR bounds: -0.1485 to 0.5090
  Z-score bounds (3 std): -0.3608 to 0.7843

  IQR Outliers: 202063 (5.76%)
  Z-score Outliers: 63900 (1.82%)


IQR (1.5 * IQR) flagged about 10.8% of the data as outliers, which is too aggressive - energy consumption is naturally right-skewed and long-tailed, so a lot of that would just be genuine high-usage days.

Z-score (3 std) was a bit better but still flagged some real winter peaks as outliers.

So instead I capped extreme values at the 99.5th percentile *per household* rather than using a single global threshold. This keeps each household's own usage pattern (a bigger house naturally uses more) while still removing the truly extreme values.


In [9]:
# Cap outliers per household at their own 99.5th percentile
print("Capping outliers at household-level 99.5th percentile...")

# 99.5th percentile per household (LCLid)
percentiles = daily.groupby('LCLid')['energy_mean'].transform(lambda x: x.quantile(0.995))

# Cap values above each household's threshold
num_capped = (daily['energy_mean'] > percentiles).sum()
daily['energy_mean'] = np.minimum(daily['energy_mean'], percentiles)

print(f"✅ Capped {num_capped} extreme values to household 99.5th percentiles.")


Capping outliers at household-level 99.5th percentile...
✅ Capped 20830 extreme values to household 99.5th percentiles.


## 5. Datetime Parsing

Before merging, I need all the date columns in the same format.


In [10]:
# Daily consumption date
daily['day'] = pd.to_datetime(daily['day'])
print(f"Daily date range: {daily['day'].min()} to {daily['day'].max()}")

# Weather date (convert 'time' to date-only datetime)
weather['time'] = pd.to_datetime(weather['time']).dt.normalize()
print(f"Weather date range: {weather['time'].min()} to {weather['time'].max()}")

# Drop duplicate weather dates to avoid a messy merge
weather = weather.drop_duplicates(subset=['time'])


Daily date range: 2011-11-23 00:00:00 to 2014-02-28 00:00:00
Weather date range: 2011-11-01 00:00:00 to 2014-03-30 00:00:00


## 6. Merging the Datasets

Merging:
1. `daily` energy consumption
2. `households` metadata (on `LCLid`)
3. `weather` daily data (on `day` = `time`)


In [11]:
# Merge daily and households
print("Merging consumption with household info...")
master_df = daily.merge(households, on='LCLid', how='inner')
print(f"  Shape after merging households: {master_df.shape}")

# Merge with weather
print("Merging with weather data...")
master_df = master_df.merge(weather, left_on='day', right_on='time', how='inner')

# Drop the redundant 'time' column from weather
master_df = master_df.drop(columns=['time'])
print(f"  Shape after merging weather: {master_df.shape}")


Merging consumption with household info...
  Shape after merging households: (3510403, 13)
Merging with weather data...
  Shape after merging weather: (3499689, 44)


## 7. Memory Optimization

3.5 million rows with mostly `float64`/`object` columns takes up a lot of RAM (over 1.5 GB). I downcasted the types to bring this down:
- `float64` to `float32`
- `int64` to `int32`/`int16` depending on the value range
- High/low-cardinality string columns (`LCLid`, `stdorToU`, `Acorn`) to `category`


In [12]:
# Check memory usage before optimizing
initial_mem = master_df.memory_usage(deep=True).sum() / 1024**2
print(f"📊 Initial Memory Usage: {initial_mem:.2f} MB")

# Downcast dtypes to save memory
for col in master_df.columns:
    col_type = master_df[col].dtype

    if col_type == 'float64':
        master_df[col] = master_df[col].astype('float32')

    elif col_type == 'int64':
        # Use a smaller int type if the value range allows it
        max_val = master_df[col].max()
        min_val = master_df[col].min()
        if max_val < 32767 and min_val > -32768:
            master_df[col] = master_df[col].astype('int16')
        else:
            master_df[col] = master_df[col].astype('int32')

    elif col_type == 'object' and col not in ['day']:
        master_df[col] = master_df[col].astype('category')

# Check memory usage after optimizing
optimized_mem = master_df.memory_usage(deep=True).sum() / 1024**2
savings = (initial_mem - optimized_mem) / initial_mem * 100
print(f"📊 Optimized Memory Usage: {optimized_mem:.2f} MB")
print(f"📉 Total Memory Savings: {savings:.2f}%")

# Garbage collection to free RAM
gc.collect()


📊 Initial Memory Usage: 4768.13 MB
📊 Optimized Memory Usage: 438.61 MB
📉 Total Memory Savings: 90.80%


0

In [13]:
# Final dataset inspection
print("First 3 rows of master dataset:")
master_df.head(3)


First 3 rows of master dataset:


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min,stdorToU,...,temperatureHigh,sunriseTime,temperatureHighTime,uvIndexTime,summary,temperatureLowTime,apparentTemperatureMin,apparentTemperatureMaxTime,apparentTemperatureLowTime,moonPhase
0,MAC000131,2011-12-15,0.4850,0.415549,0.868,22,0.239146,9.505000,0.072,Std,...,7.97,2011-12-15 08:00:46,2011-12-15 14:00:00,2011-12-15 11:00:00,Partly cloudy throughout the day and breezy in...,2011-12-16 08:00:00,1.07,2011-12-15 21:00:00,2011-12-16 08:00:00,0.66
1,MAC000131,2011-12-16,0.1415,0.296167,1.116,48,0.281471,14.216001,0.031,Std,...,4.53,2011-12-16 08:01:35,2011-12-16 15:00:00,2011-12-16 11:00:00,Mostly cloudy throughout the day.,2011-12-17 08:00:00,-2.65,2011-12-16 00:00:00,2011-12-17 08:00:00,0.70
2,MAC000131,2011-12-17,0.1015,0.189812,0.685,48,0.188405,9.111000,0.064,Std,...,5.35,2011-12-17 08:02:21,2011-12-17 14:00:00,2011-12-17 11:00:00,Partly cloudy throughout the day.,2011-12-18 07:00:00,-3.56,2011-12-17 15:00:00,2011-12-18 06:00:00,0.73


## 8. Saving the Cleaned Data

CSV doesn't keep the dtypes I just optimized - categories go back to plain objects and float32 becomes float64 again when reloaded, and the file is much bigger (~1.1 GB).

Parquet keeps all the dtypes, is compressed automatically, and is only about 120 MB.

I saved both: `master_daily.parquet` for actual modeling, and `master_daily.csv` as a plain fallback.


In [14]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

parquet_path = os.path.join(output_dir, 'master_daily.parquet')
csv_path = os.path.join(output_dir, 'master_daily.csv')

# Save as Parquet
print("Saving as Parquet...")
master_df.to_parquet(parquet_path, index=False)
print(f"✅ Saved to: {parquet_path}")

# Save as CSV (zipped to save space in Colab/Github)
print("Saving as compressed CSV...")
master_df.to_csv(csv_path + '.gz', index=False, compression='gzip')
print(f"✅ Saved to: {csv_path}.gz")

# Print file size comparison
parquet_size = os.path.getsize(parquet_path) / 1024**2
csv_size = os.path.getsize(csv_path + '.gz') / 1024**2
print(f"\n📁 Disk Space Comparison:")
print(f"  Parquet size: {parquet_size:.2f} MB")
print(f"  Zipped CSV size: {csv_size:.2f} MB")


Saving as Parquet...
✅ Saved to: ../data/processed/master_daily.parquet
Saving as compressed CSV...
✅ Saved to: ../data/processed/master_daily.csv.gz

📁 Disk Space Comparison:
  Parquet size: 66.17 MB
  Zipped CSV size: 337.55 MB


In [15]:
import os

print("Current directory:", os.getcwd())
print("Parquet:", os.path.abspath(parquet_path))
print("CSV:", os.path.abspath(csv_path + '.gz'))

Current directory: /content
Parquet: /data/processed/master_daily.parquet
CSV: /data/processed/master_daily.csv.gz


## 9. Summary

| Metric | Before | After | What I did |
|---|---|---|---|
| Household missing values | Some nulls | 0 nulls | Dropped rows with missing demographics |
| Weather missing values | Nulls in cloudCover | 0 nulls | Linear interpolation |
| Daily consumption nulls | Nulls in energy_std | 0 nulls | Filled with 0.0 (variance of 1 reading = 0) |
| Outliers (energy_mean) | Extreme peaks present | Capped | Capped at 99.5th percentile per household |
| Memory usage | ~1.5 GB | ~280 MB | Downcasted to float32 / category / int16 |
| Merged shape | 3 separate files | 1 master file | Inner merge on LCLid and date |

### Next (Notebook 3)
With a clean master dataset ready, next I build features - time-based features (weekend, holiday, season), lags, rolling stats, and weather-based features.
